# YOLOv8 v3 Li-only Binary Kurgan Detection

Kaggle notebook for the clean v3 experiment:

- source dataset: `dataset_yolo_bbox` added as a Kaggle Dataset input;
- modality: `Li` only;
- positive bbox classes: `kurgany_tselye`, `kurgany_povrezhdennye`;
- target: single binary class `kurgan`;
- positive filters: `valid_fraction >= 0.9`, `bbox_area_px` between `240` and `435600`, `bbox_touches_tile_edge == False`, `n_objects <= 20`;
- negative filters: `Li`, `valid_fraction >= 0.9`, `n_objects == 0`;
- split: inherited region-based `train/val` split from `metadata.csv`;
- model: `yolov8n.pt`;
- training: `imgsz=640`, `epochs=100`, `single_cls=True`, `close_mosaic=10`;
- output: zipped training and validation artifacts in `/kaggle/working`.

Before running, attach the Kaggle Dataset that contains either `dataset_yolo_bbox/` or `dataset_yolo_bbox.zip`.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/MataNerdy/Geodata_Archaeology_CV.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/Geodata_Archaeology_CV")
PROJECT_DIR = REPO_DIR / "04_detection_yolo"

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_DATA_ROOT = Path("/kaggle/working/datasets")
SOURCE_DATASET_NAME = "dataset_yolo_bbox"
V3_DATASET_DIR = WORK_DATA_ROOT / "dataset_yolo_bbox_v3_li_kurgan_binary"

RUN_NAME = "kurgans_li_v3_yolov8n_binary_img640_kaggle"
RUN_PROJECT = Path("/kaggle/working/runs/detect")

MODEL = "yolov8n.pt"
IMGSZ = 640
EPOCHS = 100
BATCH = 16
PATIENCE = 25
WORKERS = 2
CACHE = True
SINGLE_CLS = True
CLOSE_MOSAIC = 10

CONF_THRESHOLDS = [0.10, 0.15, 0.25]
INCLUDE_WEIGHTS_IN_ARCHIVE = True

print("Repo:", REPO_URL)
print("Input root:", KAGGLE_INPUT_ROOT)
print("Working dataset:", V3_DATASET_DIR)
print("Run name:", RUN_NAME)

## Install Dependencies and Load Repository

In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pandas", "pyyaml", "pillow"],
    check=True,
)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("CWD:", Path.cwd())
print("Project files:")
print("\n".join(sorted(p.name for p in PROJECT_DIR.iterdir())[:30]))

## Locate or Unpack Source Dataset

In [ ]:
import zipfile

def find_source_dataset(input_root: Path, dataset_name: str) -> Path:
    candidates = []
    for meta in input_root.rglob("metadata.csv"):
        parent = meta.parent
        if parent.name == dataset_name and (parent / "images").exists() and (parent / "labels").exists():
            candidates.append(parent)
    if candidates:
        return sorted(candidates, key=lambda p: len(str(p)))[0]

    zip_candidates = sorted(input_root.rglob(f"{dataset_name}.zip"))
    if zip_candidates:
        WORK_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        zip_path = zip_candidates[0]
        print("Unzipping:", zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(WORK_DATA_ROOT)
        extracted = WORK_DATA_ROOT / dataset_name
        if extracted.exists():
            return extracted

    raise FileNotFoundError(
        f"Could not find {dataset_name}/metadata.csv or {dataset_name}.zip under {input_root}. "
        "Attach the source YOLO dataset as a Kaggle Dataset input."
    )

SOURCE_DATASET_DIR = find_source_dataset(KAGGLE_INPUT_ROOT, SOURCE_DATASET_NAME)
print("Source dataset:", SOURCE_DATASET_DIR)
print("Metadata:", SOURCE_DATASET_DIR / "metadata.csv")

## Build v3 Dataset

In [ ]:
import random
import shutil
import pandas as pd
import yaml

RANDOM_SEED = 42
MIN_VALID_FRACTION = 0.9
MIN_BBOX_AREA_PX = 240
MAX_BBOX_AREA_PX = 435600
MAX_OBJECTS = 20
KEEP_MODALITIES = {"Li"}
POSITIVE_CLASS_NAMES = {"kurgany_tselye", "kurgany_povrezhdennye"}
CLASS_ID_MAP = {0: 0, 1: 0}
NAMES = {0: "kurgan"}


def resolve_dataset_path(path_value, data_root: Path, kind: str, split: str) -> Path:
    path = Path(str(path_value))
    if path.exists():
        return path
    candidate = data_root / kind / split / path.name
    if candidate.exists():
        return candidate
    candidate = data_root / str(path_value)
    if candidate.exists():
        return candidate
    return path


def clean_positive_records(same_image: pd.DataFrame):
    objects = same_image[same_image["class_id"].notna()].copy()
    objects = objects[objects["class_name"].isin(POSITIVE_CLASS_NAMES)].copy()
    objects = objects[objects["bbox_area_px"].astype(float).between(MIN_BBOX_AREA_PX, MAX_BBOX_AREA_PX)].copy()
    edge_mask = objects["bbox_touches_tile_edge"].astype("boolean").fillna(False)
    objects = objects[~edge_mask].copy()

    records = []
    for _, obj in objects.iterrows():
        old_cls = int(float(obj["class_id"]))
        if old_cls not in CLASS_ID_MAP:
            continue
        tile_size = float(obj.get("tile_size", 1024))
        x1, y1 = float(obj["bbox_x1_px"]), float(obj["bbox_y1_px"])
        x2, y2 = float(obj["bbox_x2_px"]), float(obj["bbox_y2_px"])
        xc = ((x1 + x2) / 2.0) / tile_size
        yc = ((y1 + y2) / 2.0) / tile_size
        w = (x2 - x1) / tile_size
        h = (y2 - y1) / tile_size
        if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
            continue
        records.append({"box": (CLASS_ID_MAP[old_cls], xc, yc, w, h), "source": obj.to_dict()})
    return records


def write_label(path: Path, records):
    lines = [
        f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
        for cls, xc, yc, w, h in [record["box"] for record in records]
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


def write_dataset_yaml(out_dir: Path):
    data = {
        "path": str(out_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": NAMES,
    }
    (out_dir / "dataset.yaml").write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")


random.seed(RANDOM_SEED)
if V3_DATASET_DIR.exists():
    shutil.rmtree(V3_DATASET_DIR)
for split in ["train", "val"]:
    (V3_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (V3_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

source_meta = pd.read_csv(SOURCE_DATASET_DIR / "metadata.csv")
source_images = source_meta.drop_duplicates("image").copy()
source_images = source_images[source_images["modality"].isin(KEEP_MODALITIES)].copy()
source_images = source_images[source_images["valid_fraction"].astype(float) >= MIN_VALID_FRACTION].copy()

selected = []
for _, row in source_images.iterrows():
    split = str(row["split"])
    old_img = resolve_dataset_path(row["image"], SOURCE_DATASET_DIR, "images", split)
    old_lbl = resolve_dataset_path(row["label"], SOURCE_DATASET_DIR, "labels", split)
    if not old_img.exists():
        continue

    same_image = source_meta[source_meta["image"] == row["image"]]
    original_n_objects = int(row["n_objects"])
    records = []
    is_positive = False

    if original_n_objects == 0:
        is_positive = False
    elif original_n_objects <= MAX_OBJECTS:
        records = clean_positive_records(same_image)
        is_positive = bool(records)
    else:
        continue

    if not is_positive and original_n_objects != 0:
        continue

    selected.append({
        "row": row,
        "split": split,
        "old_img": old_img,
        "old_lbl": old_lbl,
        "records": records,
        "is_positive": is_positive,
    })

new_rows = []
for item in selected:
    row = item["row"]
    split = item["split"]
    records = item["records"]
    new_img = V3_DATASET_DIR / "images" / split / item["old_img"].name
    new_lbl = V3_DATASET_DIR / "labels" / split / item["old_lbl"].name
    shutil.copy2(item["old_img"], new_img)
    write_label(new_lbl, records)

    base = row.to_dict()
    base["image"] = str(new_img)
    base["label"] = str(new_lbl)
    base["source_n_objects"] = int(row["n_objects"])
    base["is_positive"] = bool(records)
    base["n_objects"] = len(records) if records else int(row["n_objects"])

    if records:
        for record in records:
            cls_id, xc, yc, w, h = record["box"]
            source_obj = record["source"]
            obj = base.copy()
            obj.update({
                "bbox_x1_px": source_obj.get("bbox_x1_px"),
                "bbox_y1_px": source_obj.get("bbox_y1_px"),
                "bbox_x2_px": source_obj.get("bbox_x2_px"),
                "bbox_y2_px": source_obj.get("bbox_y2_px"),
                "bbox_area_px": source_obj.get("bbox_area_px"),
                "bbox_touches_tile_edge": source_obj.get("bbox_touches_tile_edge"),
                "source_class_id": source_obj.get("class_id"),
                "source_class_name": source_obj.get("class_name"),
                "class_id": cls_id,
                "class_name": NAMES[cls_id],
                "yolo_xc": xc,
                "yolo_yc": yc,
                "yolo_w": w,
                "yolo_h": h,
            })
            new_rows.append(obj)
    else:
        obj = base.copy()
        obj.update({"class_id": None, "class_name": None, "yolo_xc": None, "yolo_yc": None, "yolo_w": None, "yolo_h": None})
        new_rows.append(obj)

pd.DataFrame(new_rows).to_csv(V3_DATASET_DIR / "metadata.csv", index=False)
write_dataset_yaml(V3_DATASET_DIR)

experiment_config = {
    "variant": "v3_clean_li_binary_kurgan",
    "source_dataset": str(SOURCE_DATASET_DIR),
    "output_dataset": str(V3_DATASET_DIR),
    "modalities": sorted(KEEP_MODALITIES),
    "positive_class_names": sorted(POSITIVE_CLASS_NAMES),
    "new_classes": NAMES,
    "min_valid_fraction": MIN_VALID_FRACTION,
    "min_bbox_area_px": MIN_BBOX_AREA_PX,
    "max_bbox_area_px": MAX_BBOX_AREA_PX,
    "bbox_touches_tile_edge": False,
    "max_objects": MAX_OBJECTS,
    "negative_filter": "modality == Li and valid_fraction >= 0.9 and n_objects == 0",
    "random_seed": RANDOM_SEED,
}
(Path("/kaggle/working") / "dataset_v3_experiment_config.yaml").write_text(
    yaml.safe_dump(experiment_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

DATA_YAML = V3_DATASET_DIR / "dataset.yaml"
images = pd.DataFrame(new_rows).drop_duplicates("image")
print("Images copied:", len(images))
print("Positive images:", int(images["is_positive"].sum()))
print("Negative images:", int((~images["is_positive"]).sum()))
print("Boxes:", int(pd.DataFrame(new_rows)["is_positive"].sum()))
print("Data YAML:", DATA_YAML)
print(DATA_YAML.read_text())


## Dataset Sanity Check

In [ ]:
import pandas as pd

meta = pd.read_csv(V3_DATASET_DIR / "metadata.csv")
images = meta.drop_duplicates("image").copy()
train = images[images["split"] == "train"]
val = images[images["split"] == "val"]

display(images.groupby(["split", "modality", "is_positive"]).size().rename("images").reset_index())
display(meta[meta["is_positive"]].groupby("class_name").size().rename("boxes").reset_index())

train_regions = set(train["region"].astype(str))
val_regions = set(val["region"].astype(str))
print("Region overlap:", len(train_regions & val_regions))
print("Train regions:", len(train_regions))
print("Val regions:", len(val_regions))
print("Objects per positive image:")
print(images[images["is_positive"]]["n_objects"].describe())

## Train YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
train_result = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=PATIENCE,
    single_cls=SINGLE_CLS,
    close_mosaic=CLOSE_MOSAIC,
    cos_lr=True,
    workers=WORKERS,
    cache=CACHE,
    project=str(RUN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
)

RUN_DIR = Path(train_result.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("Run dir:", RUN_DIR)
print("Best weights:", BEST_WEIGHTS, BEST_WEIGHTS.exists())

## Validate Confidence Thresholds

In [ ]:
threshold_run_dirs = []

if BEST_WEIGHTS.exists():
    val_model = YOLO(str(BEST_WEIGHTS))
    for conf in CONF_THRESHOLDS:
        val_name = f"{RUN_NAME}_val_conf_{str(conf).replace('.', '_')}"
        metrics = val_model.val(
            data=str(DATA_YAML),
            imgsz=IMGSZ,
            conf=conf,
            single_cls=SINGLE_CLS,
            project=str(RUN_PROJECT),
            name=val_name,
            exist_ok=True,
        )
        out_dir = RUN_PROJECT / val_name
        threshold_run_dirs.append(out_dir)
        print("conf:", conf, "save_dir:", out_dir)
else:
    print("No best weights found; threshold validation skipped.")

## Summarize Training Metrics

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    results = pd.read_csv(results_csv)
    results.columns = [c.strip() for c in results.columns]
    metric_col = "metrics/mAP50(B)"
    best_idx = results[metric_col].idxmax() if metric_col in results.columns else results.index[-1]
    best_row = results.loc[[best_idx]]
    display(best_row)
else:
    print("results.csv not found:", results_csv)

## Visual Artifacts

In [ ]:
from IPython.display import Image, display

for name in [
    "results.png",
    "confusion_matrix.png",
    "PR_curve.png",
    "val_batch0_labels.jpg",
    "val_batch0_pred.jpg",
]:
    path = RUN_DIR / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

## Archive Results

In [ ]:
from datetime import datetime
import zipfile

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = Path("/kaggle/working") / f"yolo_kurgan_detection_v3_li_binary_{timestamp}.zip"

paths_to_archive = [
    RUN_DIR,
    V3_DATASET_DIR / "dataset.yaml",
    V3_DATASET_DIR / "metadata.csv",
    Path("/kaggle/working") / "dataset_v3_experiment_config.yaml",
    PROJECT_DIR / "configs" / "dataset_v3.yaml",
    PROJECT_DIR / "configs" / "train_v3_yolov8n.yaml",
] + threshold_run_dirs

def should_skip(path: Path) -> bool:
    if INCLUDE_WEIGHTS_IN_ARCHIVE:
        return False
    return path.suffix.lower() == ".pt"

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in paths_to_archive:
        path = Path(path)
        if not path.exists():
            continue
        if path.is_dir():
            for file in path.rglob("*"):
                if file.is_file() and not should_skip(file):
                    zf.write(file, arcname=file.relative_to(Path("/kaggle/working")))
        elif not should_skip(path):
            zf.write(path, arcname=path.name)

print("Archive saved:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / (1024 * 1024), 2))